# Explaining Nearest-Neighbor Classifiers: Weighted and Thresholded Classifiers

In the [previous notebook](explainers_1.ipynb) we only considered unweighted $k$-nearest neighbor classifiers.
Next, let's focus on the slightly more advanced _weighted_ $k$-nearest neighbor classifier.

> Note: Please make sure to read the previous notebook first, since it explains the basic concepts and introduces some mathematical notation.

## Weighted $k$-Nearest Neighbor Classification

Weighted $k$-nearest neighbor (WKNN) classifiers work similarly to their unweighted counterparts, but assign a weight $w_i$ to each training data point $x_i$, which should be antitone with respect to the distance $d(x_i, x)$ from the point $x$ being predicted, meaning that points farther away have lower weights. A common choice for the weight function is $w_i := \frac{1}{d(x_i, x)}$. This is also the only weight function we support.

When calculating the probability of predicting some class $c \in C$, the model then considers the weighted count of training points with class $c$ among the $k$-nearest neighbors of $x$:
$$
    \hat y = \underset{c\in C}{\text{argmax}} \sum_{j=1}^{k} w_{\alpha_j} \, \chi(y_{\alpha_j}=c)
$$

where $\alpha_j$ is the index of the $j$-nearest training data point to $x$. The advantage of WKNN is that it is generally more accurate since it's better at handling varying densities in the training data, distant but numerically dominant classes, as well as noise.

Next, we will discuss how to design a utility function for efficiently calculating Shapley values for WKNN models. If you're only interested in the actual code, you can of course skip this section and jump right into [explaining WKNN predictions](#explaining-wknn-predictions).

### Making WKNN Shapley-friendly

In order to make WKNN models more suitable for efficiently computing Shapley values, some adjustments are made compared to unweighted KNN models [\[Wng24\]](../citations.rst):

- The task of explaining a prediction $y_\text{explain}$ for a multi-class model is reduced to combining the explanations for the binary prediction of '$y_\text{explain}$ versus $c$' for all $c \neq y_\text{train}$:
  $$
    \nu(S) = \frac{1}{|C|-1} \sum_{c\in C \setminus \{y_\text{explain}\}} \nu_c(S),
  $$
  where $\nu_c$ is the utility function for the binary classificiation sub-task, as defined below.

- We consider a binary utility function, meaning that for a non-empty coalition $\emptyset \neq S \subseteq D$, we define its utility in predicting $y_\text{explain}$ versus $c$ as
  $$
    \nu_c(S) = \chi \left[
        \sum_{j=1}^{k} w_{\alpha_{S,j,c}} \, \chi(y_{\alpha_{S,j,c}} = y_\text{explain})
        \geq
        \sum_{j=1}^{k} w_{\alpha_{S,j,c}} \, \chi(y_{\alpha_{S,j,c}} = c)
    \right],
  $$
  and set $\nu(\emptyset) = 0$. Note that here we define $\alpha_{S,j,c}$ to be the index of the $j$-nearest training point to $x_\text{explain}$ among all points in $D$ that are **relevant to the current binary prediction task** '$y_\text{explain}$ versus $c$', that is, all $x_i$ for which $y_i \in \{c, y_\text{explain} \}$.

- The weights $w_i$ are normalized to the range $[0, 1]$ and then discretized to $b$ bits, meaning they are rounded to the nearest of $2^b + 1$ equally spaced values in the interval $[0, 1]$. This parameter $b$ allows the user to make a trade-off between explanation accuracy and performance.

### Explaining WKNN predictions

Let's write some actual code! As before, we will start by generating a synthetic classification dataset and plotting it.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

from plot import plot_datasets

X_train, y_train = make_classification(
    n_samples=40,
    n_features=2,
    n_redundant=0,
    n_clusters_per_class=1,
    n_informative=2,
    n_classes=2,
    random_state=49,
    # Starkly imbalanced classes, which are a good use case for WKNN
    weights=[0.9, 0.1],
)

fig, ax = plt.subplots(figsize=(6, 6))
plot_datasets(ax, X_train, y_train)
print(f"Size of training dataset: {X_train.shape[0]}")

Now we'll train a WKNN model on this training data. We will use `sklearn`'s `KNeighborsClassifier` again, but this time set the `weights` parameter to `"distance"`:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model_wknn = KNeighborsClassifier(n_neighbors=7, weights="distance")
model_wknn.fit(X_train, y_train)

Now, we let the model predict the class of a test data point.

In [ ]:
import numpy as np

x_test = np.array([[0.3, -1.4]])
y_test_pred_wknn = model_wknn.predict(x_test)[0]
print(y_test_pred_wknn)
y_test_pred_wknn_proba = model_wknn.predict_proba(x_test)[0]
print(y_test_pred_wknn_proba)

To create an explainer for the model, we can again simply pass it to `KNNExplainer` which will select the right explainer for a weighted model.

In [ ]:
from shapiq_student import KNNExplainer
from shapiq_student.explainer.knn import interaction_values_to_array

explainer_wknn = KNNExplainer(model_wknn, class_index=y_test_pred_wknn)
explainer_wknn.__class__

Now we can explain the prediction for `x_test`:

In [ ]:
sv_wknn = interaction_values_to_array(explainer_wknn.explain(x_test))
print(sv_wknn)

Let's visualize the explanation!

In [ ]:
import matplotlib.pyplot as plt

from shapiq_student.plot import plot_points_shapley_2d

fig, ax = plt.subplots(figsize=(6, 6))
plot_points_shapley_2d(ax, X_train, y_train, sv_wknn, set(y_train), x_test)

Looking good! Close points with the predicted class have large positive Shapley values, while close points with a different class have large negative values. Points farther away have values close to zero, meaning they barely influenced the prediction.

Next, let's move on to threshold nearest neighbor models.

## Threshold Nearest Neighbor Classification

When making a prediction for a data point $x_\text{test}$, threshold nearest neighbor (TNN) models consider all data points within a certain radius $\tau$ around $x_\text{test}$. Smaller values of $\tau$ are useful for detecting local structures in the training data, while larger values help to make the model less susceptible to noise.

### Explaining TNN Predictions

The `scikit-learn` library provides an implementation in the `RadiusNeighborsClassifier` class. Let's train a model, make a prediction, and explain it.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

from plot import plot_datasets

X_train, y_train = make_classification(
    n_samples=100,
    n_features=2,
    n_redundant=0,
    n_clusters_per_class=1,
    n_informative=2,
    n_classes=2,
    # The clusters of both classes are close to each other, which is a strength of TNN models
    class_sep=0.9,
    random_state=51,
)

fig, ax = plt.subplots(figsize=(6, 6))
plot_datasets(ax, X_train, y_train)
print(f"Size of training dataset: {X_train.shape[0]}")

In [ ]:
from sklearn.neighbors import RadiusNeighborsClassifier

x_test = np.array([[-0.3, -0.6]])
tau = 0.5

model = RadiusNeighborsClassifier(radius=tau)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(x_test)
print(y_pred)

As before, we create an explainer by instantiating the `KNNExplainer` class with our model.

> Note: Although the base class has the word 'KNN' in its name, it can also serve as an explainer for TNN models, which don't use any parameter $k$.

In [ ]:
explainer = KNNExplainer(model, class_index=y_pred)
sv = interaction_values_to_array(explainer.explain(x_test))
print(sv)

As we can see, a large portion of the Shapley values is zero. To understand what this means, let's plot them like before. Additionally, we'll mark the radius $\tau$ of the TNN model in the plot.

In [ ]:
from matplotlib.patches import Circle

fig, ax = plt.subplots(figsize=(6, 6))
plot_points_shapley_2d(ax, X_train, y_train, sv, set(y_train), x_test, min_size=0.1)
ax.add_patch(
    Circle(
        tuple(x_test[0]),
        tau,
        facecolor="none",
        edgecolor="black",
        linestyle="--",
    )
);

Here we can see where all those zeros come from: All training data points outside of the $\tau$-neighbourhood around $x_\text{test}$ have zero Shapley values. This is because they can't influence the prediction at all. (Note that in the plot, values of zero are represented as dots of size `0.1` since we set `min_size=0.1`. We do this so that they still remain visible.)

That's it for this notebook! For detailed information on the API of the explainer classes, see the [API reference](../api/shapiq_student.explainer.rst).